# Baseline RAG Pipeline
## Overview
This notebook contains the baseline implementation of a Retrieval-Augmented Generation (RAG) pipeline. This is the first approach, using simple FAISS retrieval without any query routing, query expansion, reranking, or conversational memory.

## Contents
- Baseline RAG pipeline implementation
- Predefined question test cell
- Interactive test cell
- Evaluation across 3 embedding models and 3 generation models
  - Embedding models: MiniLM-L6-v2, BGE-base-en-v1.5, Multi-QA-MPNet-base
  - Generation models: Mistral-7B-v0.1, Mistral-7B-Instruct-v0.2, Phi-2

### 1.Baseline RAG pipeline implementation

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")

from dotenv import load_dotenv
import os
load_dotenv()
token = os.environ.get("HF_TOKEN")

# ============================================================================
# BASELINE RAG PIPELINE CLASS 
# ============================================================================

class BaselineRAGPipeline:
    """
    Simple baseline RAG pipeline with same interface as advanced models
    Uses basic FAISS retrieval without any advanced matchers or logic
    """
    
    def __init__(self, model, tokenizer, embedding_model_name, faiss_index, chunks, generation_model_name=None):
        self.model = model
        self.tokenizer = tokenizer
        self.embedding_model_name = embedding_model_name
        self.faiss_index = faiss_index
        self.chunks = chunks
        self.generation_model_name = generation_model_name  # ← NEW
        self.use_llm = model is not None and tokenizer is not None
    
    def retrieve_contexts(self, query: str, k: int = 5):
        """Simple FAISS retrieval"""
        query_embedding = self.embedding_model_name.encode([query])
        distances, indices = self.faiss_index.search(query_embedding, k)
        return [self.chunks[i] for i in indices[0]]
    
    def extract_answer_from_context(self, question: str, context: str) -> str:
        """Extract answer directly from context without LLM"""
        question_lower = question.lower()
        lines = context.split('\n')
        
        # Look for relevant lines
        relevant_lines = []
        for line in lines:
            line_lower = line.lower()
            question_words = [w for w in question_lower.split() if len(w) > 3]
            if any(word in line_lower for word in question_words):
                relevant_lines.append(line)
        
        if relevant_lines:
            return '\n'.join(relevant_lines[:3])
        
        return "I couldn't find a specific answer in the retrieved context."
    
    def generate_llm_answer(self, question: str, context: str) -> str:
        """Generate answer using LLM"""
        if not self.use_llm:
            return self.extract_answer_from_context(question, context)
        
        # Model-specific prompt formatting
        model_name = (self.generation_model_name or "").lower()
        
        if "instruct-v0.2" in model_name:
            prompt = f"[INST] Answer based only on the context below. Be concise and direct.\n\nContext:\n{context}\n\nQuestion: {question} [/INST]"
        elif "phi-2" in model_name:
            prompt = f"Instruct: Answer based only on the context below. Be concise and direct.\n\nContext:\n{context}\n\nQuestion: {question}\nOutput:"
        else:  # Mistral-v0.1 and any other base models
            prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
        
        try:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                output = self.model.generate(
                    **inputs,
                    max_new_tokens=150,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    do_sample=False,
                    repetition_penalty=1.1
                )
            
            full_response = self.tokenizer.decode(output[0], skip_special_tokens=True)
            
            # Extract answer
            answer_markers = ["[/INST]", "Output:", "Answer:"]
            answer = full_response
            
            for marker in answer_markers:
                if marker in full_response:
                    answer = full_response.split(marker)[-1].strip()
                    break
            
            # Clean up
            if "Context:" in answer:
                answer = answer.split("Context:")[0].strip()
            if "Question:" in answer:
                answer = answer.split("Question:")[0].strip()
            
            lines = answer.split('\n')
            clean_lines = []
            for line in lines:
                line = line.strip()
                if line and not line.startswith(('Context:', 'Question:', 'Answer:', 'Instruct:', 'Output:')):
                    clean_lines.append(line)
                elif clean_lines:
                    break
            
            answer = '\n'.join(clean_lines).strip()
            
            if not answer:
                answer = self.extract_answer_from_context(question, context)
            
            return answer
        
        except Exception as e:
            print(f"   LLM generation error: {e}")
            return self.extract_answer_from_context(question, context)
    
    def answer_query(self, query: str, verbose: bool = False) -> str:
        """
        Main method to answer queries - same interface as advanced models
        
        Args:
            query: User question
            verbose: Print debug information
            
        Returns:
            Generated answer string
        """
        if verbose:
            print(f"\nQuestion: {query}")
            print("-" * 70)
        
        # Retrieve contexts
        relevant_chunks = self.retrieve_contexts(query, k=5)
        context = "\n\n".join(relevant_chunks)
        
        if verbose:
            print(f"\n--- Retrieved {len(relevant_chunks)} chunks ---")
            for i, chunk in enumerate(relevant_chunks, 1):
                print(f"Chunk {i}:")
                print(chunk[:200] + "..." if len(chunk) > 200 else chunk)
                print()
        
        # Generate answer
        if self.use_llm:
            answer = self.generate_llm_answer(query, context)
        else:
            answer = self.extract_answer_from_context(query, context)
        
        if verbose:
            print(f"\nAnswer: {answer}")
            print("-" * 70)
        
        return answer


# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_all_data(course_data_file="data/all_programs_data.json",
                 professor_data_file="data/hof_university_staff.json",
                 university_data_file="data/general_hof_university_data.json",
                 aspo_data_file="data/aspo_priority_sections.json"):
    """Load all data sources"""
    
    print("\n1. Loading all data sources...")
    
    # Load course data
    try:
        with open(course_data_file, 'r', encoding='utf-8') as f:
            course_data = json.load(f)
        print(f"   Loaded {len(course_data)} programs")
    except FileNotFoundError:
        print(f" Course data not found: {course_data_file}")
        course_data = {}
    
    # Load professor data
    try:
        with open(professor_data_file, 'r', encoding='utf-8') as f:
            professor_data = json.load(f)
        print(f"   Loaded {len(professor_data)} professors")
    except FileNotFoundError:
        print(f" Professor data not found: {professor_data_file}")
        professor_data = []
    
    # Load university data
    try:
        with open(university_data_file, 'r', encoding='utf-8') as f:
            university_data = json.load(f)
        print(f"   Loaded university general data")
    except FileNotFoundError:
        print(f" University data not found: {university_data_file}")
        university_data = {}
    
    # Load ASPO data
    try:
        with open(aspo_data_file, 'r', encoding='utf-8') as f:
            aspo_data = json.load(f)
        print(f"   Loaded ASPO examination regulations")
    except FileNotFoundError:
        print(f" ASPO data not found: {aspo_data_file}")
        aspo_data = {}
    
    return course_data, professor_data, university_data, aspo_data


def create_simple_chunks(course_data, professor_data, university_data, aspo_data):
    """Create simple text chunks from all data sources"""
    
    print("\n2. Creating simple text chunks from all data sources...")
    chunks = []
    
    # --- PROGRAM CHUNKS ---
    for program_name, details in course_data.items():
        # Basic program info chunk
        chunk = f"Program: {program_name}\n"
        
        if details.get('program_type'):
            chunk += f"Type: {details['program_type']}\n"
        if details.get('department'):
            chunk += f"Department: {details['department']}\n"
        if details.get('degree'):
            chunk += f"Degree: {details['degree']}\n"
        if details.get('language_of_instruction'):
            chunk += f"Language: {details['language_of_instruction']}\n"
        if details.get('duration'):
            chunk += f"Duration: {details['duration']}\n"
        if details.get('campus'):
            chunk += f"Campus: {details['campus']}\n"
        if details.get('start'):
            chunk += f"Start: {details['start']}\n"
        if details.get('tuition_fees'):
            chunk += f"Tuition Fees: {details['tuition_fees']}\n"
        if details.get('eligible_backgrounds'):
            chunk += f"Eligible Backgrounds: {', '.join(details['eligible_backgrounds'])}\n"
        
        chunks.append(chunk)
        
        # Admission requirements chunk
        if details.get('admission_requirements'):
            req_chunk = f"Program: {program_name}\nAdmission Requirements:\n"
            for key, value in details['admission_requirements'].items():
                if value:
                    req_chunk += f"{key}: {value}\n"
            chunks.append(req_chunk)
        
        # Application link chunk
        if details.get('application_link'):
            app_chunk = f"Program: {program_name}\nApplication Link: {details['application_link']}\n"
            chunks.append(app_chunk)
        
        # Contact chunk
        if details.get('contact'):
            contact_chunk = f"Program: {program_name}\nContact Information:\n"
            for key, value in details['contact'].items():
                if value:
                    contact_chunk += f"{key}: {value}\n"
            chunks.append(contact_chunk)
        
        # Module chunks
        if details.get('modules'):
            for module in details['modules']:
                module_chunk = f"Program: {program_name}\n"
                module_chunk += f"Module: {module.get('module_name', 'N/A')}\n"
                module_chunk += f"ECTS: {module.get('ects', 'N/A')}\n"
                module_chunk += f"Exam Type: {module.get('exam_type', 'N/A')}\n"
                module_chunk += f"Course Type: {module.get('course_type', 'N/A')}\n"
                chunks.append(module_chunk)
        
        # Thesis chunk
        if details.get('thesis'):
            thesis_chunk = f"Program: {program_name}\nThesis Requirements:\n"
            for key, value in details['thesis'].items():
                if value:
                    thesis_chunk += f"{key}: {value}\n"
            chunks.append(thesis_chunk)
    
    # --- PROFESSOR CHUNKS ---
    for prof in professor_data:
        prof_chunk = f"Professor: {prof.get('name', 'N/A')}\n"
        prof_chunk += f"Title: {prof.get('title', 'N/A')}\n"
        prof_chunk += f"Department: {prof.get('department', 'N/A')}\n"
        prof_chunk += f"Email: {prof.get('email', 'N/A')}\n"
        prof_chunk += f"Office: {prof.get('office', 'N/A')}\n"
        prof_chunk += f"Phone: {prof.get('phone', 'N/A')}\n"
        
        if prof.get('programs'):
            prof_chunk += f"Programs: {', '.join(prof['programs'])}\n"
        
        chunks.append(prof_chunk)
    
    # --- UNIVERSITY DATA CHUNKS ---
    for category, info in university_data.items():
        if isinstance(info, dict):
            uni_chunk = f"Category: {category}\n"
            for key, value in info.items():
                if value:
                    uni_chunk += f"{key}: {value}\n"
            chunks.append(uni_chunk)
        elif isinstance(info, list):
            for item in info:
                if isinstance(item, dict):
                    uni_chunk = f"Category: {category}\n"
                    for key, value in item.items():
                        uni_chunk += f"{key}: {value}\n"
                    chunks.append(uni_chunk)
                else:
                    uni_chunk = f"Category: {category}\n{item}\n"
                    chunks.append(uni_chunk)
        else:
            uni_chunk = f"Category: {category}\n{info}\n"
            chunks.append(uni_chunk)
    
    # --- ASPO CHUNKS ---
    for section_name, section_content in aspo_data.items():
        if isinstance(section_content, dict):
            aspo_chunk = f"ASPO Section: {section_name}\n"
            
            if section_content.get('title'):
                aspo_chunk += f"Title: {section_content['title']}\n"
            
            if section_content.get('full_text'):
                aspo_chunk += f"\n{section_content['full_text']}\n"
            
            if section_content.get('key_points'):
                aspo_chunk += "\nKey Points:\n"
                for point in section_content['key_points']:
                    aspo_chunk += f"- {point}\n"
            
            chunks.append(aspo_chunk)
    
 
    
    return chunks


def initialize_complete_system(course_data_file="data/all_programs_data.json",
                               professor_data_file="data/hof_university_staff.json",
                               university_data_file="data/general_hof_university_data.json",
                               aspo_data_file="data/aspo_priority_sections.json",
                               embedding_model_name='all-MiniLM-L6-v2',      
                               generation_model_name='mistralai/Mistral-7B-v0.1',  
                               load_llm=True,
                               token=None):
    """
    Initialize the baseline RAG system
    
    Returns:
        BaselineRAGPipeline object with answer_query() method
    """
    

    # Load all data
    course_data, professor_data, university_data, aspo_data = load_all_data(
        course_data_file, professor_data_file, university_data_file, aspo_data_file
    )
    
    if not course_data:
        print("No course data loaded. Cannot initialize system.")
        return None
    
    # Create chunks
    chunks = create_simple_chunks(course_data, professor_data, university_data, aspo_data)
    
    # Create embeddings
    print("\n3. Creating embeddings...")
    embedding_model_name = SentenceTransformer(embedding_model_name)
    embeddings = embedding_model_name.encode(chunks, show_progress_bar=True)
    print(f"   Embeddings shape: {embeddings.shape}")
    
    # Build FAISS index
    print("\n4. Building FAISS index...")
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatL2(dimension)
    faiss_index.add(np.array(embeddings).astype('float32'))
    print(f"   Index built with {faiss_index.ntotal} vectors")
    
    # Load LLM (optional)
    model, tokenizer = None, None
    
    if load_llm:
        print("\n" )
        print("5. Loading LLM model")
        print()

        print(f"   Model: {generation_model_name}")
        
        
        if token is None:
            token = os.environ.get("HF_TOKEN")
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"   Using device: {device}")
        
        try:
            if device == "cuda":
                print("   Loading with 4-bit quantization...")
                bnb_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True
                )
                
                tokenizer = AutoTokenizer.from_pretrained(generation_model_name, token=token)
                model = AutoModelForCausalLM.from_pretrained(
                    generation_model_name,
                    quantization_config=bnb_config,
                    device_map="auto",
                    token=token,
                    attn_implementation="eager",  # Disable Flash Attention
                    trust_remote_code=True
                )
            else:
                print("Loading on CPU...")
                tokenizer = AutoTokenizer.from_pretrained(generation_model_name, token=token)
                model = AutoModelForCausalLM.from_pretrained(
                    generation_model_name,
                    token=token,
                    torch_dtype=torch.float32,
                    attn_implementation="eager",  # Disable Flash Attention
                    trust_remote_code=True
                )
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            print("   Model loaded successfully")
        
        except Exception as e:
            print(f" Model loading failed: {e}")
            print("   Continuing without LLM...")
            model, tokenizer = None, None
    else:
        print("\nLLM loading skipped (load_llm=False)")
    
    rag_pipeline = BaselineRAGPipeline(
        model=model,
        tokenizer=tokenizer,
        embedding_model_name=embedding_model_name,
        faiss_index=faiss_index,
        chunks=chunks,
        generation_model_name=generation_model_name  # ← ADD THIS
    )
    

    
    return rag_pipeline


# ============================================================================
# TEST THE SYSTEM
# ============================================================================

if __name__ == "__main__":
    # Initialize baseline system
    rag_pipeline = initialize_complete_system(
        course_data_file="data/all_programs_data.json",
        professor_data_file="data/hof_university_staff.json",
        university_data_file="data/general_hof_university_data.json",
        aspo_data_file="data/aspo_priority_sections.json",
        embedding_model_name='all-MiniLM-L6-v2',
        generation_model_name='mistralai/Mistral-7B-v0.1',
        load_llm=True,
        token=None
    )
    
    if rag_pipeline:
        # Test with a few questions
        test_questions = [
            "I studied engineering, what bachelor programs can I apply to?",
            "I studied engineering, what Masters programs I can apply?",
            "I studied Management, what Masters programs I can apply?",
            "Programs available for economics background?",
          

 
        ]
        
        print("\n")
        print("Testing Baseline System...")
        print("\n")
        
        for i, question in enumerate(test_questions, 1):
            print(f"\n--- Test {i}/{len(test_questions)} ---")
            answer = rag_pipeline.answer_query(question, verbose=False)
            print(f"Q: {question}")
            print(f"A: {answer}")
            print("-" * 70)


1. Loading all data sources...
   Loaded 37 programs
   Loaded 175 professors
   Loaded university general data
   Loaded ASPO examination regulations

2. Creating simple text chunks from all data sources...

3. Creating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

   Embeddings shape: (333, 384)

4. Building FAISS index...
   Index built with 333 vectors


5. Loading LLM model

   Model: mistralai/Mistral-7B-v0.1
   Using device: cuda
   Loading with 4-bit quantization...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

   Model loaded successfully


Testing Baseline System...



--- Test 1/4 ---
Q: I studied engineering, what bachelor programs can I apply to?
A: Type: Bachelor
Type: Bachelor
----------------------------------------------------------------------

--- Test 2/4 ---
Q: I studied engineering, what Masters programs I can apply?
A: I couldn't find a specific answer in the retrieved context.
----------------------------------------------------------------------

--- Test 3/4 ---
Q: I studied Management, what Masters programs I can apply?
A: I couldn't find a specific answer in the retrieved context.
----------------------------------------------------------------------

--- Test 4/4 ---
Q: Programs available for economics background?
A: Module: Digital Economics
Module: Applied Economics and Intercultural Management
----------------------------------------------------------------------


### 2. Predefined question test cell

In [5]:
# ============================================================================
# TEST CELL
# ============================================================================
test_questions = [
    "I studied engineering, what bachelor programs can I apply to?",
    "I studied engineering, what Masters programs I can apply?",
    "I studied Management, what Masters programs I can apply?",
    "Programs available for economics background?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\nQuestion {i}: {question}")
    answer = rag_pipeline.answer_query(question, verbose=False)
    print(f"Answer: {answer}")
    print("-" * 50)


Question 1: I studied engineering, what bachelor programs can I apply to?
Answer: The bachelor program in Environmental Engineering would be a good fit for you! It is a general program with a strong focus on sustainability and environmental protection.
--------------------------------------------------

Question 2: I studied engineering, what Masters programs I can apply?
Answer: Please note that this is an automated answer.
--------------------------------------------------

Question 3: I studied Management, what Masters programs I can apply?
Answer: I couldn't find a specific answer in the retrieved context.
--------------------------------------------------

Question 4: Programs available for economics background?
Answer: In Germany, there are many programs available for economics background. These include economic psychology, general management, digital economics, applied econometrics, and intercultural management. However, there may be additional requirements such as a bachelor's

### 3. Interactive test cell

In [6]:
# ============================================================================
# INTERACTIVE CHAT
# ============================================================================
print("Chat started. Type 'quit' to exit.\n")

while True:
    query = input("You: ").strip()
    if not query:
        continue
    if query.lower() in ("quit", "exit"):
        break
    answer = rag_pipeline.answer_query(query, verbose=False)
    print(f"Answer: {answer}\n")

Chat started. Type 'quit' to exit.



You:  I studied engineering, what bachelor programs can I apply to?


Answer: The engineering science program is a good option, since it has a more general focus than the other two programs listed above. However, if you have specific interests, you may want to consider applying to one of those programs instead.



You:  exit


### 4. Evaluation across 3 embedding models and 3 generation models
#### - Embedding models: MiniLM-L6-v2, BGE-base-en-v1.5, Multi-QA-MPNet-base
#### - Generation models: Mistral-7B-v0.1, Mistral-7B-Instruct-v0.2, Phi-2

In [3]:
# ============================================================================
# CELL 1: IMPORTS AND EVALUATOR CLASS
# ============================================================================

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import json
from datetime import datetime
import time
import re
import os
import torch
from typing import Dict, List, Set, Tuple

class RAGEvaluator:
    def __init__(self, embedding_model_name='all-MiniLM-L6-v2'):
        """Initialize the enhanced RAG evaluator with embedding model"""
        print("Loading embedding model for evaluation...")
        self.embedding_model_name = SentenceTransformer(embedding_model_name)
        print("Embedding model loaded!")
    
    def extract_entities(self, text: str) -> Set[str]:
        """Extract numbers, dates, URLs, proper nouns (single + multi-word), and important words"""
        text_lower = text.lower()
        entities = set()
        
        # Numbers
        entities.update(re.findall(r'\b\d+\.?\d*\b', text))
        
        # Dates
        entities.update(re.findall(r'\b\d{1,2}[./]\d{1,2}[./]\d{2,4}\b', text))
        
        # URLs
        entities.update(re.findall(r'https?://[^\s]+', text))
        
        # Proper nouns (single + multi-word)
        entities.update(re.findall(r'\b[A-Z][a-z]+\b', text))  # single-word
        entities.update(re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+\b', text))  # multi-word
        
        # Important words (5+ characters, excluding common words)
        common_words = {'what', 'when', 'where', 'which', 'there', 'their', 'would', 
                       'could', 'should', 'about', 'these', 'those', 'that', 'this', 
                       'with', 'from', 'have', 'been', 'were', 'they', 'will', 'your',
                       'more', 'than', 'then', 'them', 'some', 'other', 'into', 'only',
                       'such', 'make', 'over', 'very', 'even', 'most', 'also', 'after',
                       'before', 'through', 'being', 'under', 'while'}
        words = re.findall(r'\b[a-z]{5,}\b', text_lower)
        entities.update([w for w in words if w not in common_words])
        
        return set(e.lower() for e in entities)

        
    
    def precision_score(self, expected: str, actual: str) -> float:
        expected_entities = self.extract_entities(expected)
        actual_entities = self.extract_entities(actual)
        
        if len(actual_entities) == 0:
            return 0.0
        
        true_positives = len(expected_entities.intersection(actual_entities))
        return float(true_positives / len(actual_entities))
    
    def recall_score(self, expected: str, actual: str) -> float:
        expected_entities = self.extract_entities(expected)
        actual_entities = self.extract_entities(actual)
        
        if len(expected_entities) == 0:
            return 1.0
        
        true_positives = len(expected_entities.intersection(actual_entities))
        return float(true_positives / len(expected_entities))
    
    def f1_score(self, expected: str, actual: str) -> float:
        precision = self.precision_score(expected, actual)
        recall = self.recall_score(expected, actual)
        
        if precision + recall == 0:
            return 0.0
        
        return float(2 * (precision * recall) / (precision + recall))

    def entity_f1(self, expected: str, actual: str) -> float:
        precision = self.precision_score(expected, actual)
        recall = self.recall_score(expected, actual)
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)

    def token_f1(self, expected: str, actual: str) -> float:
        exp_tokens = set(expected.lower().split())
        act_tokens = set(actual.lower().split())
        if len(exp_tokens) == 0 and len(act_tokens) == 0:
            return 1.0
        intersection = exp_tokens & act_tokens
        precision = len(intersection) / len(act_tokens) if len(act_tokens) > 0 else 0.0
        recall = len(intersection) / len(exp_tokens) if len(exp_tokens) > 0 else 0.0
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)  
    
    def cosine_similarity_score(self, expected: str, actual: str) -> float:
        expected_emb = self.embedding_model_name.encode([expected])
        actual_emb = self.embedding_model_name.encode([actual])
        similarity = cosine_similarity(expected_emb, actual_emb)[0][0]
        return float(similarity)
    
    def semantic_similarity_score(self, expected: str, actual: str) -> float:
        expected_emb = self.embedding_model_name.encode(expected, convert_to_tensor=True)
        actual_emb = self.embedding_model_name.encode(actual, convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(expected_emb, actual_emb).item()
        return float(similarity)
    
    def token_overlap_score(self, expected: str, actual: str) -> float:
        expected_tokens = set(expected.lower().split())
        actual_tokens = set(actual.lower().split())
        
        if len(expected_tokens) == 0 and len(actual_tokens) == 0:
            return 1.0
        
        intersection = expected_tokens.intersection(actual_tokens)
        union = expected_tokens.union(actual_tokens)
        
        return len(intersection) / len(union) if len(union) > 0 else 0.0
    
    def answer_relevance_score(self, question: str, answer: str) -> float:
        question_emb = self.embedding_model_name.encode(question, convert_to_tensor=True)
        answer_emb = self.embedding_model_name.encode(answer, convert_to_tensor=True)
        relevance = util.pytorch_cos_sim(question_emb, answer_emb).item()
        return float(relevance)
    
    def hallucination_score(self, expected: str, actual: str) -> float:
        expected_entities = self.extract_entities(expected)
        actual_entities = self.extract_entities(actual)
        
        if len(actual_entities) == 0:
            return 0.0
        
        false_positives = len(actual_entities - expected_entities)
        return float(false_positives / len(actual_entities))
    
    def faithfulness_score(self, expected: str, actual: str) -> float:
        semantic_sim = self.semantic_similarity_score(expected, actual)
        hallucination = self.hallucination_score(expected, actual)
        key_info_coverage = 1.0 - hallucination
        
        expected_length = len(expected.split())
        actual_length = len(actual.split())
        
        if expected_length == 0:
            length_ratio = 1.0
        else:
            length_ratio = actual_length / expected_length
        
        if length_ratio <= 1.5:
            length_penalty = 1.0
        elif length_ratio <= 2.0:
            length_penalty = 0.9
        elif length_ratio <= 3.0:
            length_penalty = 0.7
        else:
            length_penalty = 0.5
        
        faithfulness = (semantic_sim * 0.4 + key_info_coverage * 0.6) * length_penalty
        return float(faithfulness)
    
    def answer_completeness(self, expected: str, actual: str) -> float:
        expected_sentences = [s.strip() for s in expected.split('.') if s.strip()]
        actual_sentences = [s.strip() for s in actual.split('.') if s.strip()]
        
        if not expected_sentences:
            return 1.0
        if not actual_sentences:
            return 0.0
        
        covered = 0
        for exp_sent in expected_sentences:
            exp_emb = self.embedding_model_name.encode(exp_sent, convert_to_tensor=True)
            max_similarity = 0.0
            
            for act_sent in actual_sentences:
                act_emb = self.embedding_model_name.encode(act_sent, convert_to_tensor=True)
                sim = util.pytorch_cos_sim(exp_emb, act_emb).item()
                max_similarity = max(max_similarity, sim)
            
            if max_similarity > 0.7:
                covered += 1
        
        return covered / len(expected_sentences)
    
    def contains_key_information(self, expected: str, actual: str) -> float:
        expected_numbers = set(re.findall(r'\d+', expected))
        actual_numbers = set(re.findall(r'\d+', actual))
        
        expected_caps = set(re.findall(r'\b[A-Z][a-z]+\b', expected))
        actual_caps = set(re.findall(r'\b[A-Z][a-z]+\b', actual))
        
        expected_urls = set(re.findall(r'https?://[^\s]+', expected))
        actual_urls = set(re.findall(r'https?://[^\s]+', actual))
        
        scores = []
        
        if expected_numbers:
            number_score = len(expected_numbers.intersection(actual_numbers)) / len(expected_numbers)
            scores.append(number_score)
        
        if expected_caps:
            caps_score = len(expected_caps.intersection(actual_caps)) / len(expected_caps)
            scores.append(caps_score)
        
        if expected_urls:
            url_score = len(expected_urls.intersection(actual_urls)) / len(expected_urls)
            scores.append(url_score)
        
        return np.mean(scores) if scores else 0.5
    
    def exact_match_score(self, expected: str, actual: str) -> float:
        return 1.0 if expected.strip().lower() == actual.strip().lower() else 0.0
    
    def calculate_all_metrics(self, question: str, expected: str, actual: str, 
                             latency: float = None) -> Dict[str, float]:
        metrics = {
            'precision': self.precision_score(expected, actual),
            'recall': self.recall_score(expected, actual),
            'entity_f1': self.entity_f1(expected, actual),
            'token_f1': self.token_f1(expected, actual),
            'cosine_similarity': self.cosine_similarity_score(expected, actual),
            'semantic_similarity': self.semantic_similarity_score(expected, actual),
            'token_overlap': self.token_overlap_score(expected, actual),
            'answer_relevance': self.answer_relevance_score(question, actual),
            'hallucination_score': self.hallucination_score(expected, actual),
            'faithfulness': self.faithfulness_score(expected, actual),
            'answer_completeness': self.answer_completeness(expected, actual),
            'key_information_coverage': self.contains_key_information(expected, actual),
            'exact_match': self.exact_match_score(expected, actual),
        }
        
        if latency is not None:
            metrics['latency_seconds'] = latency
        
        return metrics

print("Evaluator class loaded!")

Evaluator class loaded!


In [4]:
# ============================================================================
# CELL 2: EVALUATION FUNCTION
# ============================================================================

def evaluate_single_model(rag_pipeline, test_csv_path: str, model_name: str, 
                         output_dir: str = './evaluation_results'):
    """Evaluate a single RAG pipeline configuration"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"EVALUATING: {model_name.upper()}")
    print(f"{'='*80}\n")
    
    # Load test cases
    print(f"Loading test cases from {test_csv_path}...")
    test_df = pd.read_csv(test_csv_path)
    print(f"Loaded {len(test_df)} test cases\n")
    
    # Initialize evaluator
    evaluator = RAGEvaluator()
    
    # Store results
    results = []
    
    print("Running evaluation...")
    print("-" * 80)
    
    # Evaluate each test case
    for idx, row in test_df.iterrows():
        question = row['question']
        expected_answer = row['expected_answer']
        
        print(f"[{idx + 1}/{len(test_df)}] {question[:60]}...")
        
        # Get actual answer and measure latency
        start_time = time.time()
        actual_answer = rag_pipeline.answer_query(question)
        latency = time.time() - start_time
        
        # Calculate all metrics
        metrics = evaluator.calculate_all_metrics(question, expected_answer, 
                                                   actual_answer, latency)
        
        # Store result
        result = {
            'model': model_name,
            'test_id': idx + 1,
            'question': question,
            'expected_answer': expected_answer,
            'actual_answer': actual_answer,
            'Entity-F1': metrics['entity_f1'],     # Changed
            'Token-F1': metrics['token_f1'],       # Changed
            'Cosine Similarity': metrics['cosine_similarity'],
            'Semantic Similarity': metrics['semantic_similarity'],
            'Answer Relevance': metrics['answer_relevance'],
            'Hallucination': metrics['hallucination_score'],
            'Faithfulness': metrics['faithfulness'],
            'Completeness': metrics['answer_completeness'],
            'Key Info': metrics['key_information_coverage'],
            'Exact Match': metrics['exact_match'],
            'Latency (s)': metrics.get('latency_seconds', None)
        }
        results.append(result)
    
    # Create results DataFrame
    results_df = pd.DataFrame(results)

    gen_quality = (results_df['Entity-F1'].mean() + results_df['Token-F1'].mean() + results_df['Semantic Similarity'].mean()) / 3
    faithfulness = (results_df['Faithfulness'].mean() + (1 - results_df['Hallucination'].mean()) + results_df['Key Info'].mean()) / 3
    completeness = results_df['Completeness'].mean()
    
    overall_score = 0.4 * gen_quality + 0.35 * faithfulness + 0.25 * completeness


    
    # Print summary
    print("\n" + "="*80)
    print("EVALUATION SUMMARY")
    print("="*80)
    print(f"Model: {model_name}")
    print(f"Overall Score: {overall_score:.4f}")
    print(f"Entity-F1: {results_df['Entity-F1'].mean():.4f}")
    print(f"Token-F1: {results_df['Token-F1'].mean():.4f}")
    print(f"Semantic Similarity: {results_df['Semantic Similarity'].mean():.4f}")
    print(f"Answer Relevance: {results_df['Answer Relevance'].mean():.4f}")
    print(f"Faithfulness: {results_df['Faithfulness'].mean():.4f}")
    print(f"Hallucination: {results_df['Hallucination'].mean():.4f}")
    print(f"Completeness: {results_df['Completeness'].mean():.4f}")
    print(f"Avg Latency: {results_df['Latency (s)'].mean():.2f}s")
    print("="*80 + "\n")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = f"{output_dir}/{model_name}_results.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Results saved: {csv_path}\n")
    
    # Create summary
    summary = {
        'model': model_name,
        'timestamp': timestamp,
        'total_test_cases': len(test_df),
        'overall_score': float(overall_score),
        'Entity-F1': float(results_df['Entity-F1'].mean()),
        'Token-F1': float(results_df['Token-F1'].mean()),
        'Semantic Similarity': float(results_df['Semantic Similarity'].mean()),
        'Answer Relevance': float(results_df['Answer Relevance'].mean()),
        'Faithfulness': float(results_df['Faithfulness'].mean()),
        'Hallucination': float(results_df['Hallucination'].mean()),
        'Completeness': float(results_df['Completeness'].mean()),
        'Latency (s)': float(results_df['Latency (s)'].mean())
    }
    
    return results_df, summary

print("Evaluation function loaded!")

Evaluation function loaded!


In [ ]:


# ============================================================================
# CELL 3: RUN ALL EVALUATIONS (3 × 3 = 9 CONFIGURATIONS)
# ============================================================================

# Define model configurations
EMBEDDINGS = {
    'minilm': 'all-MiniLM-L6-v2',
    'bge': 'BAAI/bge-base-en-v1.5',
    'multiqa': 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
}

GENERATIONS = {
    'mistral_v01': 'mistralai/Mistral-7B-v0.1',
    'mistral_v02': 'mistralai/Mistral-7B-Instruct-v0.2',
    'phi2': 'microsoft/phi-2'
}

# Configuration
TEST_CSV = './test_cases_100.csv'  # Change this to your test file
OUTPUT_DIR = './evaluation_results/baseline'
token = None  # Add your token if needed

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "="*80)
print("STARTING COMPREHENSIVE EVALUATION")
print("="*80)
print(f"Embedding Models: {len(EMBEDDINGS)}")
print(f"Generation Models: {len(GENERATIONS)}")
print(f"Total Configurations: {len(EMBEDDINGS) * len(GENERATIONS)}")
print("="*80 + "\n")

all_summaries = []
config_num = 0
total_configs = len(EMBEDDINGS) * len(GENERATIONS)

start_time = time.time()

# Loop through all combinations
for emb_name, emb_model in EMBEDDINGS.items():
    for gen_name, gen_model in GENERATIONS.items():
        config_num += 1
        
        model_id = f"baseline_{emb_name}_{gen_name}"
        
        print("\n" + "="*80)
        print(f"CONFIGURATION {config_num}/{total_configs}")
        print("="*80)
        print(f"Model ID: {model_id}")
        print(f"Embedding: {emb_model}")
        print(f"Generation: {gen_model}")
        print("="*80 + "\n")
        
        try:
            # Initialize RAG system (ASSUMING YOUR initialize_complete_system EXISTS)
            print(f"Initializing system...")
            rag_pipeline = initialize_complete_system(
                embedding_model_name=emb_model,
                generation_model_name=gen_model,
                load_llm=True
            
            )
            
            if not rag_pipeline:
                print(f"Failed to initialize {model_id}")
                continue
            
            # Run evaluation
            results_df, summary = evaluate_single_model(
                rag_pipeline=rag_pipeline,
                test_csv_path=TEST_CSV,
                model_name=model_id,
                output_dir=OUTPUT_DIR
            )
            
            # Add configuration info to summary
            summary['embedding_model_name'] = emb_name
            summary['embedding_name'] = emb_model
            summary['generation_model_name'] = gen_name
            summary['generation_name'] = gen_model
            
            all_summaries.append(summary)
            
            print(f"Completed: {model_id}")
            
            # Clean up GPU memory
            del rag_pipeline
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"Error with {model_id}: {e}")
            continue

total_time = time.time() - start_time

print("ALL EVALUATIONS COMPLETED!")
print(f"Total Time: {total_time/60:.1f} minutes")



STARTING COMPREHENSIVE EVALUATION
Embedding Models: 3
Generation Models: 3
Total Configurations: 9


CONFIGURATION 1/9
Model ID: baseline_minilm_mistral_v01
Embedding: all-MiniLM-L6-v2
Generation: mistralai/Mistral-7B-v0.1

Initializing system...

1. Loading all data sources...
   Loaded 37 programs
   Loaded 175 professors
   Loaded university general data
   Loaded ASPO examination regulations

2. Creating simple text chunks from all data sources...

3. Creating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

   Embeddings shape: (333, 384)

4. Building FAISS index...
   Index built with 333 vectors


5. Loading LLM model

   Model: mistralai/Mistral-7B-v0.1
   Using device: cuda
   Loading with 4-bit quantization...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

   Embeddings shape: (333, 768)

4. Building FAISS index...
   Index built with 333 vectors


5. Loading LLM model

   Model: microsoft/phi-2
   Using device: cuda
   Loading with 4-bit quantization...


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

   Model loaded successfully

EVALUATING: BASELINE_MULTIQA_PHI2

Loading test cases from ./test_cases_100.csv...
Loaded 100 test cases

Loading embedding model for evaluation...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!
Running evaluation...
--------------------------------------------------------------------------------
[1/100] I have a computer science background, which master's courses...
[2/100] What programs are available for business students?...
[3/100] I studied engineering, what bachelor programs can I apply to...
[4/100] I studied engineering, what Masters programs I can apply?...
[5/100] Programs available for economics background?...
[6/100] I have a law background, what can I study?...
[7/100] I studied Management, what Masters programs I can apply?...
[8/100] What programs can psychology students apply for?...
[9/100] I have a design background, what master programs are availab...
[10/100] Programs for textile technology students?...
[11/100] I studied IT, which programs can I apply for?...
[12/100] I studied informatics, what master programs can I apply to?...
[13/100] I have a Science background, what bachelor programs can I ap...
[14/100] I have a Business back